In [1]:
!kaggle datasets download umairimran627/voice-dataset-for-audio-deep-fake

Dataset URL: https://www.kaggle.com/datasets/umairimran627/voice-dataset-for-audio-deep-fake
License(s): MIT
100% 5.81G/5.81G [01:14<00:00, 83.7MB/s]



In [2]:
import zipfile

In [3]:
zip_path = "/content/voice-dataset-for-audio-deep-fake.zip"
extract_path = "/content/voice"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted!")

Dataset extracted!


In [4]:
!pip install tensorflow -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.9/572.9 MB 788.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 91.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 72.1 MB/s eta 0:00:00


In [5]:
import os
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt


import tensorflow as tf
from tensorflow.keras import layers, models

/usr/local/lib/python3.13/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


In [6]:
BASE_PATH = "/content/voice/for-norm"

TRAIN_PATH = BASE_PATH + "/training"
VAL_PATH   = BASE_PATH + "/validation"
TEST_PATH  = BASE_PATH + "/testing"

In [7]:
def extract_mel_spectrogram(file_path):

    audio, sr = librosa.load(file_path, sr=16000)

    # 4 seconds
    max_length = 16000 * 4

    # Padding
    if len(audio) < max_length:
        audio = np.pad(
            audio,
            (0, max_length - len(audio))
        )

    # Cutting
    else:
        audio = audio[:max_length]

    # Mel spectrogram
    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=sr,
        n_mels=128
    )

    # Convert to dB
    mel_db = librosa.power_to_db(
        mel,
        ref=np.max
    )

    return mel_db

In [8]:
def load_dataset(dataset_path):

    X = []
    y = []

    classes = {
        "fake": 0,
        "real": 1
    }

    for class_name, label in classes.items():

        class_path = os.path.join(
            dataset_path,
            class_name
        )

        for file_name in os.listdir(class_path):

            file_path = os.path.join(
                class_path,
                file_name
            )

            if file_name.lower().endswith(
                (".wav", ".mp3", ".flac", ".ogg")
            ):

                try:

                    mel = extract_mel_spectrogram(
                        file_path
                    )

                    X.append(mel)
                    y.append(label)

                except Exception as e:

                    print(
                        "Error:",
                        file_name,
                        e
                    )

    return np.array(X), np.array(y)

In [9]:
X_train, y_train = load_dataset(TRAIN_PATH)

X_val, y_val = load_dataset(VAL_PATH)

X_test, y_test = load_dataset(TEST_PATH)

In [10]:
print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Testing:", X_test.shape, y_test.shape)

Training: (53868, 128, 126) (53868,)
Validation: (10798, 128, 126) (10798,)
Testing: (4634, 128, 126) (4634,)


In [11]:
X_train = X_train[..., np.newaxis]
X_val = X_val[..., np.newaxis]
X_test = X_test[..., np.newaxis]

In [12]:
print(X_train.shape)

(53868, 128, 126, 1)


In [13]:
model = models.Sequential([

    layers.Input(
        shape=X_train.shape[1:]
    ),

    layers.Conv2D(
        32,
        (3, 3),
        activation="relu"
    ),

    layers.MaxPooling2D(
        (2, 2)
    ),

    layers.Conv2D(
        64,
        (3, 3),
        activation="relu"
    ),

    layers.MaxPooling2D(
        (2, 2)
    ),

    layers.Conv2D(
        128,
        (3, 3),
        activation="relu"
    ),

    layers.MaxPooling2D(
        (2, 2)
    ),

    layers.Flatten(),

    layers.Dense(
        128,
        activation="relu"
    ),

    layers.Dropout(0.5),

    layers.Dense(
        1,
        activation="sigmoid"
    )
])

In [14]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [15]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 124, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 62, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 60, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     3,211,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,304,193 (12.60 MB)

 Trainable params: 3,304,193 (12.60 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
history = model.fit(

    X_train,
    y_train,

    validation_data=(
        X_val,
        y_val
    ),

    epochs=2,
    batch_size=256
)

Epoch 1/2
211/211 ━━━━━━━━━━━━━━━━━━━━ 571s 3s/step - accuracy: 0.8195 - loss: 1.1085 - val_accuracy: 0.8672 - val_loss: 0.3382
Epoch 2/2
211/211 ━━━━━━━━━━━━━━━━━━━━ 569s 3s/step - accuracy: 0.9065 - loss: 0.2459 - val_accuracy: 0.9429 - val_loss: 0.1531


In [17]:
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test
)

print("Test Accuracy:", test_accuracy)

145/145 ━━━━━━━━━━━━━━━━━━━━ 11s 79ms/step - accuracy: 0.8418 - loss: 0.3766
Test Accuracy: 0.8418213129043579


In [18]:
model.save(
    "audio_deepfake_detector.keras"
)